# F3: lattice-image (R) resolution (implemented)

**Finding** (`notes/theory_audit.md`, F3): crawfish summed $\mathrm{Re}[T^*_u T_v]$ over k with no Bloch phase,
which gives the R = 0 element. "Atom A – atom B" therefore meant the copy of B in A's cell, which depends on how
the input wrapped the atoms.

**Fix:** pair spectra take `image="nearest"` (default), `"all"`, or a lattice vector. For each atom pair they use
$M_{uv}(R)\,e^{2\pi i k_t\cdot R}$, with $M(R)$ the Fourier sum of the state-resolved matrices (notes section 2.3).
`get_pair_images` reports the R and bond length used. **Legacy switch:** `legacy_lattice_image=True` makes
"nearest" mean R = (0,0,0) for every pair.

This notebook has three parts:
1. A 1D dimerized chain (SSH model) through crawfish's kernels: the R=0 kernel (legacy), the R-resolved kernels
   used by the fix (`get_mat_r_suv`, `_get_gen_tj_block`), and an independent reference.
2. The real N2 data, described with atom 2 wrapped one cell over (`fixcompare.wrapped_edata` applies the Bloch-sum
   gauge change $T \to e^{2\pi i k\cdot L}T$). Legacy and fixed crawfish are compared on both descriptions, each with
   later fixes held at legacy (`load_edata_at`).
3. Graphene (2 C, 23×23×1 k-mesh): a real periodic calculation where two of each atom's three nearest neighbours
   are in neighbouring cells, so inter-cell pairs carry most of the C1–C2 bonding.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))  # fixcompare.py lives next to this notebook (run with this directory as cwd)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fixcompare import *
from crawfish.funcs.general import SIGMA_DEFAULT
from crawfish.core.operations.matrix import (
    _get_gen_tj_block, get_h_s_d_t_uv, get_mat_r_suv,
)
from crawfish.funcs import get_ipcohp, get_pcohp, get_pair_images

F3_DONE = fix_available("F3")
print("F3 implemented:", F3_DONE)

## Part 1: dimerized chain

In [ ]:
nk = 60
ks = np.arange(nk) / nk
ks_t = np.stack([ks, np.zeros(nk), np.zeros(nk)], axis=1)  # k along the chain, reciprocal-lattice coordinates
wk = np.full(nk, 2.0 / nk, dtype=np.float32)  # spin-unpolarized JDFTx convention: weights sum to 2
T_SHORT, T_LONG = -1.0, -0.3                  # hoppings (Ha) for the 0.2 and 0.8 bonds
occ = np.zeros((nk, 2), dtype=np.float32)
occ[:, 0] = 1.0                               # half filling: lower band full


def chain(tau_B):
    """Projections (orthonormal TB orbitals: T = eigenvectors) and eigenvalues; B images at tau_B + R."""
    proj = np.zeros((nk, 2, 2), dtype=np.complex64)
    e = np.zeros((nk, 2), dtype=np.float32)
    for i, k in enumerate(ks):
        hab = sum(t * np.exp(2j * np.pi * k * R) for R in range(-2, 3)
                  for d, t in [(0.2, T_SHORT), (0.8, T_LONG)] if np.isclose(abs(tau_B + R), d))
        ev, C = np.linalg.eigh(np.array([[0, hab], [np.conj(hab), 0]]))
        e[i], proj[i] = ev, C.T  # proj[t, j, u] = C_uj
    return proj, e


def nearest_R(tau_B):
    return min((-1, 0, 1), key=lambda R: abs(tau_B + R))


def crawfish_weights(proj, e, R, mode):
    """crawfish's R-resolved path: M(R) from the state-resolved matrix, phase exp(2 pi i k.R), block kernel."""
    if mode == "como":
        mat_suv = np.ones((1, 2, 2), dtype=np.complex64)
    else:
        h_t_uv, _, _ = get_h_s_d_t_uv(proj, proj, e, occ)
        mat_suv = get_mat_r_suv(h_t_uv, wk, ks_t, 1, (R, 0, 0))
    phase_t = np.exp(2j * np.pi * ks * R)
    mat_t_ab = (mat_suv[0, 0, 1] * phase_t)[:, None, None]
    return _get_gen_tj_block(proj, mat_t_ab, wk, [0], [1])


def reference_weights(proj, tau_B, R, mode):
    """Independent reference: the hopping of that image is known exactly for the model."""
    d = abs(tau_B + R)
    m = 1.0 if mode == "como" else (T_SHORT if np.isclose(d, 0.2) else T_LONG if np.isclose(d, 0.8) else 0.0)
    return wk[:, None] * (np.conj(proj[:, :, 0]) * m * np.exp(2j * np.pi * ks * R)[:, None] * proj[:, :, 1]).real


WRAPPINGS = {"B at +0.2": 0.2, "B at -0.8": -0.8}
data = {name: chain(tau) for name, tau in WRAPPINGS.items()}
rows = []
for name, tau in WRAPPINGS.items():
    proj, e = data[name]
    R = nearest_R(tau)
    for mode in ["como", "cohp"]:
        rows.append({
            "cell choice": name, "quantity": "bond order" if mode == "como" else "ICOHP (Ha)",
            "legacy (R=0)": float(np.sum(occ * crawfish_weights(proj, e, 0, mode))),
            f"fixed (nearest R)": float(np.sum(occ * crawfish_weights(proj, e, R, mode))),
            "reference (nearest R)": float(np.sum(occ * reference_weights(proj, tau, R, mode))),
            "nearest R": R,
        })
chain_table = pd.DataFrame(rows).set_index(["cell choice", "quantity"])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, q in zip(axes, ["bond order", "ICOHP (Ha)"]):
    chain_table.xs(q, level="quantity")[["legacy (R=0)", "fixed (nearest R)", "reference (nearest R)"]].plot.bar(ax=ax, rot=0)
    ax.set_title(f"A–B {q}")
    ax.axhline(0, color="black", lw=0.5)
fig.tight_layout()
savefig(fig, "F3_chain_bond_orders")
plt.show()
chain_table.round(4)

## Part 2: N2 with atom 2 wrapped into the next cell

N2 is isolated in a 13 Å box, so its nearest image is the same-cell copy. Moving atom 2 by one lattice vector
describes the same crystal, but the same-cell copy is then 12 Å away. The fixed crawfish still pairs the nearest
copy; the legacy behaviour reports the far one.

In [ ]:
L = (1, 0, 0)
descriptions = {}
for fixed in (False, True):
    tag = "fixed" if fixed else "legacy"
    descriptions[f"{tag}, original cell"] = load_edata_at("F3", fixed=fixed)
    descriptions[f"{tag}, atom 2 wrapped"] = wrapped_edata(load_edata_at("F3", fixed=fixed), "N #2", L)

for name, ed in descriptions.items():
    (pair,) = get_pair_images(ed, idcs1=0, idcs2=1)
    print(f"{name:26s} R = {pair['R']}, N-N distance = {pair['distance']:.4f} A")

PAIRS = [("s", "s"), ("s", "px"), ("px", "px"), ("py", "py"), (None, None)]
label = lambda o1, o2: "all-all" if o1 is None else f"{o1}-{o2}"
n2_table = pd.DataFrame({
    name: {label(o1, o2): last_value(get_ipcohp(ed, idcs1=0, idcs2=1, orbs1=o1, orbs2=o2, use_cache=False))
           for o1, o2 in PAIRS}
    for name, ed in descriptions.items()
})
ax = n2_table.plot.bar(rot=0, figsize=(10, 3.5))
ax.axhline(0, color="black", lw=0.5)
ax.set_ylabel("IpCOHP (Ha), negative = bonding")
savefig(ax.figure, "F3_n2_wrapping_icohp")
plt.show()
n2_table.round(5)

In [ ]:
erange = np.arange(descriptions["fixed, original cell"].e_tj.min() - 0.03,
                   descriptions["fixed, original cell"].e_tj.max() + 0.03, 0.0005, dtype=np.float32)
x_ev = to_ev_rel_mu(descriptions["fixed, original cell"], erange)
fig, ax = plt.subplots(figsize=(7, 3.5))
for name, style in [("fixed, original cell", "-"), ("fixed, atom 2 wrapped", "--"), ("legacy, atom 2 wrapped", ":")]:
    _, spec = get_pcohp(descriptions[name], idcs1=0, idcs2=1, erange=erange, sig=SIGMA_DEFAULT,
                        use_cache=False, use_cached_spectrum=False, save_spectrum=False)
    ax.plot(x_ev, -np.asarray(spec), style, label=name)
ax.set_xlabel(r"$E - \mu$ (eV)")
ax.set_ylabel("−pCOHP, N1–N2 (all orbitals)")
ax.legend()
savefig(fig, "F3_n2_wrapping_spectra")
plt.show()

## Part 3: graphene

`tests/files/io/example_calc_dirs/graphene_1x1` holds a GBRV/PBE graphene primitive cell after lattice relaxation,
with CANDLE solvation, a 23×23×1 mesh and `symmetries none`. Each C has three nearest C neighbours at about
1.456 Å. They differ by up to 0.003 Å after relaxation, so crawfish warns that "nearest" picks one of three
near-equivalent bonds. The first figure resolves the C1–C2 IpCOHP by image (shell). The second shows that
orbital-resolved values differ between the three equivalent bonds, because each bond points a different way.

In [ ]:
import itertools
import warnings
from crawfish.utils.lattice import get_image_distance

gr_fixed = load_edata_at("F3", fixed=True, calc_dir=GRAPHENE_CALC_DIR)
gr_legacy = load_edata_at("F3", fixed=False, calc_dir=GRAPHENE_CALC_DIR)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    print("fixed: ", get_pair_images(gr_fixed, idcs1=0, idcs2=1))
print("\n".join(str(w.message) for w in caught))
print("legacy:", get_pair_images(gr_legacy, idcs1=0, idcs2=1))

def ipcohp(ed, image, o1=None, o2=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # the near-tie warning was shown above
        return last_value(get_ipcohp(ed, idcs1=0, idcs2=1, orbs1=o1, orbs2=o2, image=image, use_cache=False))

shells = pd.DataFrame([
    {"R": R, "distance (A)": get_image_distance(gr_fixed, 0, 1, R), "IpCOHP (Ha)": ipcohp(gr_fixed, R)}
    for R in itertools.product(range(-11, 12), range(-11, 12), [0])  # one R per class of the 23x23 mesh
]).sort_values("distance (A)").reset_index(drop=True)
all_images = ipcohp(gr_fixed, "all")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].scatter(shells["distance (A)"], shells["IpCOHP (Ha)"], s=12)
nearest = shells.iloc[:3]
axes[0].scatter(nearest["distance (A)"], nearest["IpCOHP (Ha)"], s=60, facecolors="none", edgecolors="C3",
                label="3 nearest bonds (overlapping)")
axes[0].set_xlabel("C1–C2 image distance (Å)")
axes[0].set_ylabel("IpCOHP per image (Ha)")
axes[0].legend()
axes[1].plot(shells["distance (A)"], np.cumsum(shells["IpCOHP (Ha)"]), drawstyle="steps-post",
             label="sum of images up to d")
axes[1].axhline(all_images, color="k", ls="--", label='image="all"')
axes[1].axhline(ipcohp(gr_legacy, "nearest"), color="C3", ls=":", label="legacy (same-cell copy only)")
axes[1].set_xlabel("distance cutoff (Å)")
axes[1].set_ylabel("cumulative IpCOHP (Ha)")
axes[1].legend(fontsize=8)
for ax in axes:
    ax.set_xlim(0, 16)
fig.tight_layout()
savefig(fig, "F3_graphene_shells")
plt.show()
shells.head(10).round(5)

In [ ]:
bonds = [tuple(int(x) for x in R) for R in nearest["R"]]
orb_table = pd.DataFrame({
    f"R={R}": {**{o: ipcohp(gr_fixed, R, o, o) for o in ["s", "px", "py", "pz"]}, "all": ipcohp(gr_fixed, R)}
    for R in bonds
})
ax = orb_table.plot.bar(rot=0, figsize=(8, 3.5))
ax.axhline(0, color="black", lw=0.5)
ax.set_ylabel("IpCOHP (Ha), C1 x – C2 x")
ax.set_title("The three nearest C1–C2 bonds, orbital-resolved")
savefig(ax.figure, "F3_graphene_bond_orbitals")
plt.show()
orb_table.round(5)

Wrapping C2 into the next cell describes the same crystal. Fixed crawfish still finds a nearest bond; legacy
crawfish now pairs C1 with a second-shell copy (2.91 Å).

In [ ]:
gr_wrapped = {
    "fixed, original": gr_fixed,
    "fixed, C2 wrapped": wrapped_edata(load_edata_at("F3", fixed=True, calc_dir=GRAPHENE_CALC_DIR), "C #2", (1, 0, 0)),
    "legacy, original": gr_legacy,
    "legacy, C2 wrapped": wrapped_edata(load_edata_at("F3", fixed=False, calc_dir=GRAPHENE_CALC_DIR), "C #2", (1, 0, 0)),
}
wrap_table = pd.Series({name: ipcohp(ed, "nearest") for name, ed in gr_wrapped.items()}, name="IpCOHP (Ha)")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    wrap_dist = {name: get_pair_images(ed, idcs1=0, idcs2=1)[0]["distance"] for name, ed in gr_wrapped.items()}
ax = wrap_table.plot.bar(rot=0, figsize=(8, 3.5), color=["C0", "C0", "C3", "C3"])
for i, name in enumerate(wrap_table.index):
    ax.annotate(f"{wrap_dist[name]:.3f} Å", (i, wrap_table[name]), ha="center", va="top", fontsize=8)
ax.set_ylabel("C1–C2 IpCOHP (Ha)")
savefig(ax.figure, "F3_graphene_wrapping")
plt.show()
wrap_table.round(5)

## Tests

In [ ]:
# Chain: the fixed kernels match the reference and don't depend on the cell choice
for name in WRAPPINGS:
    for q in ["bond order", "ICOHP (Ha)"]:
        row = chain_table.loc[(name, q)]
        assert np.isclose(row["fixed (nearest R)"], row["reference (nearest R)"], atol=1e-5), (name, q)
for q in ["bond order", "ICOHP (Ha)"]:
    vals = chain_table.xs(q, level="quantity")
    assert np.isclose(*vals["fixed (nearest R)"], atol=1e-5)
    assert not np.isclose(*vals["legacy (R=0)"], atol=1e-3)  # legacy depends on the cell choice

# N2: the fixed results don't depend on the wrapping; legacy loses the bond when atom 2 is wrapped
assert np.allclose(n2_table["fixed, original cell"], n2_table["fixed, atom 2 wrapped"], atol=1e-5)
assert np.allclose(n2_table["legacy, original cell"], n2_table["fixed, original cell"], atol=1e-5)
assert abs(n2_table.loc["all-all", "legacy, atom 2 wrapped"]) < 1e-3

# Graphene: three equivalent nearest bonds, two of them inter-cell; shells beyond them still matter
assert sorted(bonds) == sorted([(-1, -1, 0), (0, -1, 0), (0, 0, 0)])
assert np.allclose(nearest["IpCOHP (Ha)"], nearest["IpCOHP (Ha)"].iloc[0], rtol=0.01)
assert all_images - nearest["IpCOHP (Ha)"].sum() < -0.05
assert np.isclose(shells["IpCOHP (Ha)"].sum(), all_images, atol=1e-5)  # Fourier identity over the 23x23 classes
assert orb_table.loc["px"].max() - orb_table.loc["px"].min() > 0.02  # m-resolved differs between bonds
assert np.isclose(wrap_table["fixed, original"], wrap_table["fixed, C2 wrapped"], atol=1e-5)
assert abs(wrap_table["legacy, C2 wrapped"]) < 0.25 * abs(wrap_table["fixed, original"])
assert F3_DONE
print("All F3 checks passed.")